# AlignForge English — Stage 3: Direct Preference Optimization (DPO)

This notebook implements closed-form Direct Preference Optimization (Rafailov et al., 2023) from first principles:
$$\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \left( \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right) \right]$$
- **Reference Model:** Frozen copy of the SFT model (no gradient updates).
- **Response Masking:** Strictly masks prompt tokens from sequence log-probabilities.

In [ ]:
import copy
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')

## 1. Primitives: Model, LoRA, and Tokenizer (1:1 from src)

In [ ]:
import json
from pathlib import Path
import re
from typing import Dict, List, Optional, Tuple, Union


# Pre-tokenization regex pattern (splitting on contractions, words, numbers, punctuation, spaces)
SPLIT_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\s\w]+|\s+"""
)


class ByteLevelBPETokenizer(ITokenizer):

    SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

    def __init__(self) -> None:
        self._special_tokens = list(self.SPECIAL_TOKENS)
        self._pad_token_id = 0
        self._unk_token_id = 1
        self._bos_token_id = 2
        self._eos_token_id = 3

        # Encapsulated state dictionaries
        self._vocab: Dict[bytes, int] = {}
        self._inverse_vocab: Dict[int, bytes] = {}
        self._merges: Dict[Tuple[bytes, bytes], bytes] = {}
        self._merge_ranks: Dict[Tuple[bytes, bytes], int] = {}

        self._init_base_vocab()

    def _init_base_vocab(self) -> None:
        self._vocab.clear()
        self._inverse_vocab.clear()

        # 1. Special tokens
        for idx, token in enumerate(self._special_tokens):
            raw = token.encode("utf-8")
            self._vocab[raw] = idx
            self._inverse_vocab[idx] = raw

        # 2. 256 byte values
        offset = len(self._special_tokens)
        for b in range(256):
            raw = bytes([b])
            self._vocab[raw] = offset + b
            self._inverse_vocab[offset + b] = raw

    # Properties implementing ITokenizer and backward compatibility
    @property
    def vocab_size(self) -> int:
        return len(self._vocab)

    @property
    def pad_id(self) -> int:
        return self._pad_token_id

    @property
    def pad_token_id(self) -> int:
        return self._pad_token_id

    @property
    def unk_token_id(self) -> int:
        return self._unk_token_id

    @property
    def bos_id(self) -> int:
        return self._bos_token_id

    @property
    def bos_token_id(self) -> int:
        return self._bos_token_id

    @property
    def eos_id(self) -> int:
        return self._eos_token_id

    @property
    def eos_token_id(self) -> int:
        return self._eos_token_id

    @property
    def special_tokens(self) -> List[str]:
        return list(self._special_tokens)

    @special_tokens.setter
    def special_tokens(self, tokens: List[str]) -> None:
        self._special_tokens = list(tokens)

    @property
    def vocab(self) -> Dict[bytes, int]:
        return self._vocab

    @vocab.setter
    def vocab(self, val: Dict[bytes, int]) -> None:
        self._vocab = val

    @property
    def inverse_vocab(self) -> Dict[int, bytes]:
        return self._inverse_vocab

    @inverse_vocab.setter
    def inverse_vocab(self, val: Dict[int, bytes]) -> None:
        self._inverse_vocab = val

    @property
    def merges(self) -> Dict[Tuple[bytes, bytes], bytes]:
        return self._merges

    @property
    def merge_ranks(self) -> Dict[Tuple[bytes, bytes], int]:
        return self._merge_ranks

    def train(self, texts: List[str], target_vocab_size: int = 2048) -> None:
        self._init_base_vocab()
        self._merges.clear()
        self._merge_ranks.clear()

        # 1. Pre-tokenize and split words into sequences of 1-byte elements
        word_counts: Dict[Tuple[bytes, ...], int] = {}
        for text in texts:
            words = SPLIT_PATTERN.findall(text)
            for w in words:
                raw_bytes = w.encode("utf-8")
                token_tuple = tuple(bytes([b]) for b in raw_bytes)
                word_counts[token_tuple] = word_counts.get(token_tuple, 0) + 1

        num_merges = target_vocab_size - len(self._vocab)

        # 2. Iteratively merge most frequent adjacent byte pairs
        for rank in range(num_merges):
            pair_counts: Dict[Tuple[bytes, bytes], int] = {}
            for word_tokens, count in word_counts.items():
                for i in range(len(word_tokens) - 1):
                    pair = (word_tokens[i], word_tokens[i + 1])
                    pair_counts[pair] = pair_counts.get(pair, 0) + count

            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            merged = best_pair[0] + best_pair[1]

            new_token_id = len(self._vocab)
            self._vocab[merged] = new_token_id
            self._inverse_vocab[new_token_id] = merged
            self._merges[best_pair] = merged
            self._merge_ranks[best_pair] = rank

            # Apply merge to all words in training set
            new_word_counts: Dict[Tuple[bytes, ...], int] = {}
            for word_tokens, count in word_counts.items():
                new_tokens = self._merge_word(word_tokens, best_pair, merged)
                new_word_counts[new_tokens] = count
            word_counts = new_word_counts

    def _merge_word(
        self,
        tokens: Tuple[bytes, ...],
        pair: Tuple[bytes, bytes],
        replacement: bytes,
    ) -> Tuple[bytes, ...]:
        out: List[bytes] = []
        i = 0
        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                out.append(replacement)
                i += 2
            else:
                out.append(tokens[i])
                i += 1
        return tuple(out)

    def _bpe(self, word_bytes: bytes) -> List[bytes]:
        tokens = [bytes([b]) for b in word_bytes]
        if len(tokens) <= 1:
            return tokens

        while len(tokens) > 1:
            best_rank = float("inf")
            best_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self._merge_ranks.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged = self._merges[best_pair]
            tokens = list(self._merge_word(tuple(tokens), best_pair, merged))

        return tokens

    def encode(
        self,
        text: str,
        add_bos: bool = False,
        add_eos: bool = False,
    ) -> List[int]:
        words = SPLIT_PATTERN.findall(text)
        token_ids: List[int] = []

        if add_bos:
            token_ids.append(self._bos_token_id)

        for w in words:
            raw = w.encode("utf-8")
            bpe_tokens = self._bpe(raw)
            for tok in bpe_tokens:
                token_id = self._vocab.get(tok, self._unk_token_id)
                token_ids.append(token_id)

        if add_eos:
            token_ids.append(self._eos_token_id)

        return token_ids

    def decode(
        self,
        ids: List[int],
        skip_special_tokens: bool = True,
    ) -> str:
        byte_chunks: List[bytes] = []
        special_ids = {
            self._pad_token_id,
            self._unk_token_id,
            self._bos_token_id,
            self._eos_token_id,
        }

        for token_id in ids:
            if skip_special_tokens and token_id in special_ids:
                continue
            raw = self._inverse_vocab.get(token_id, b"")
            byte_chunks.append(raw)

        full_bytes = b"".join(byte_chunks)
        return full_bytes.decode("utf-8", errors="replace")

    def save(self, directory: Union[str, Path]) -> None:
        dir_path = Path(directory)
        dir_path.mkdir(parents=True, exist_ok=True)

        vocab_serializable = {
            tok.hex(): idx for tok, idx in self._vocab.items()
        }
        merges_serializable = [
            [p[0].hex(), p[1].hex()] for p in self._merges.keys()
        ]

        data = {
            "vocab": vocab_serializable,
            "merges": merges_serializable,
            "special_tokens": self._special_tokens,
        }

        with open(dir_path / "tokenizer.json", "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    @classmethod
    def load(cls, directory: Union[str, Path]) -> ByteLevelBPETokenizer:
        dir_path = Path(directory)
        with open(dir_path / "tokenizer.json", "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = cls()
        tok._special_tokens = data.get("special_tokens", cls.SPECIAL_TOKENS)
        tok._vocab = {bytes.fromhex(k): v for k, v in data["vocab"].items()}
        tok._inverse_vocab = {v: k for k, v in tok._vocab.items()}

        tok._merges.clear()
        tok._merge_ranks.clear()
        for rank, (p0, p1) in enumerate(data.get("merges", [])):
            pair = (bytes.fromhex(p0), bytes.fromhex(p1))
            merged = pair[0] + pair[1]
            tok._merges[pair] = merged
            tok._merge_ranks[pair] = rank

        return tok

In [ ]:
import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        normed = x * torch.rsqrt(variance + self.eps)
        return normed * self.weight


class RotaryEmbedding(nn.Module):

    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0) -> None:
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base

        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int) -> None:
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)  # (seq_len, dim // 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if seq_len > self.cos_cached.shape[0]:
            self._build_cache(seq_len)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    position_ids: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    cos = cos.unsqueeze(0).unsqueeze(1)  # (1, 1, seq_len, head_dim)
    sin = sin.unsqueeze(0).unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout_p = dropout
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # Handle dynamic KV Cache if present
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        new_cache = (k, v) if use_cache else None
        total_k_len = k.shape[2]

        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask: only apply when decoding multiple tokens or uncached
        if T > 1:
            # Query positions: [total_k_len - T, total_k_len - 1]
            # Key positions: [0, total_k_len - 1]
            q_idx = torch.arange(total_k_len - T, total_k_len, device=x.device).unsqueeze(1)
            k_idx = torch.arange(total_k_len, device=x.device).unsqueeze(0)
            causal_mask = (k_idx > q_idx).unsqueeze(0).unsqueeze(0)  # (1, 1, T, total_k_len)
            scores = scores.masked_fill(causal_mask, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        context = torch.matmul(attn_weights, v)  # (B, n_heads, T, head_dim)
        context = context.transpose(1, 2).contiguous().view(B, T, C)

        out = self.resid_dropout(self.out_proj(context))
        return out, new_cache


class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        dropout: float = 0.1,
        norm_eps: float = 1e-5,
    ) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model, eps=norm_eps)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout)
        self.ffn_norm = RMSNorm(d_model, eps=norm_eps)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        # Pre-norm Self-Attention with residual connection
        normed_x = self.attn_norm(x)
        attn_out, new_cache = self.attn(
            normed_x, cos, sin, kv_cache=kv_cache, use_cache=use_cache
        )
        x = x + attn_out

        # Pre-norm FFN with residual connection
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_cache

from pathlib import Path
from typing import Any, List, Optional, Tuple, Union
import torch
import torch.nn as nn



class CausalTransformerModel(nn.Module, ICausalLM):

    def __init__(
        self,
        config: Optional[ModelArchitectureConfig] = None,
        vocab_size: int = 2048,
        d_model: int = 256,
        n_heads: int = 4,
        n_layers: int = 4,
        d_ff: int = 1024,
        max_seq_len: int = 2048,
        dropout: float = 0.1,
        norm_eps: float = 1e-5,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if config is not None:
            self._config = config
        else:
            self._config = ModelArchitectureConfig(
                d_model=d_model,
                n_heads=n_heads,
                n_layers=n_layers,
                d_ff=d_ff,
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
                dropout=dropout,
                eps=norm_eps,
            )

        self.vocab_size = self._config.vocab_size
        self.d_model = self._config.d_model
        self.n_heads = self._config.n_heads
        self.n_layers = self._config.n_layers
        self.d_ff = self._config.d_ff
        self.max_seq_len = self._config.max_seq_len

        # Token embedding
        self.tok_embeddings = nn.Embedding(self.vocab_size, self.d_model)
        self.drop = nn.Dropout(self._config.dropout)

        # RoPE provider
        self.rope = RotaryEmbedding(
            self.d_model // self.n_heads,
            max_seq_len=self.max_seq_len,
            base=rope_base,
        )

        # Transformer decoder blocks
        self.layers = nn.ModuleList([
            TransformerBlock(
                d_model=self.d_model,
                n_heads=self.n_heads,
                d_ff=self.d_ff,
                dropout=self._config.dropout,
                norm_eps=self._config.eps,
            )
            for _ in range(self.n_layers)
        ])

        # Final normalization & language model head
        self.norm = RMSNorm(self.d_model, eps=self._config.eps)
        self.lm_head = nn.Linear(self.d_model, self.vocab_size, bias=False)

        # Weight tying between token embeddings and LM head
        self.tok_embeddings.weight = self.lm_head.weight

        self._init_weights()

    @property
    def config(self) -> ModelArchitectureConfig:
        return self._config

    def _init_weights(self) -> None:
        for name, param in self.named_parameters():
            if "weight" in name and param.dim() >= 2:
                nn.init.normal_(param, mean=0.0, std=0.02)
            elif "bias" in name:
                nn.init.zeros_(param)

    def count_parameters(self) -> int:
        seen = set()
        total = 0
        for p in self.parameters():
            if p.requires_grad and p.data_ptr() not in seen:
                seen.add(p.data_ptr())
                total += p.numel()
        return total

    def get_num_params(self, non_embedding: bool = False) -> int:
        return self.count_parameters()

    def forward(
        self,
        input_ids: torch.Tensor,
        kv_caches: Optional[Union[list, IKVCache]] = None,
        use_cache: bool = False,
    ) -> Any:
        # Check if kv_caches is an IKVCache instance
        if isinstance(kv_caches, IKVCache):
            return self._forward_with_ikvcache(input_ids, kv_caches)

        B, T = input_ids.shape
        x = self.drop(self.tok_embeddings(input_ids))

        past_len = (
            kv_caches[0][0].shape[2]
            if (kv_caches is not None and kv_caches[0] is not None)
            else 0
        )
        total_len = past_len + T
        cos, sin = self.rope(x, total_len)
        cos = cos[past_len:total_len]
        sin = sin[past_len:total_len]

        new_caches = [] if use_cache else None

        for idx, layer in enumerate(self.layers):
            layer_cache = kv_caches[idx] if kv_caches is not None else None
            x, new_cache = layer(
                x, cos, sin, kv_cache=layer_cache, use_cache=use_cache
            )
            if use_cache:
                new_caches.append(new_cache)

        x = self.norm(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        return logits, new_caches

    def _forward_with_ikvcache(
        self, input_ids: torch.Tensor, cache: IKVCache
    ) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.drop(self.tok_embeddings(input_ids))

        past_len = cache.current_seq_len
        total_len = past_len + T
        cos, sin = self.rope(x, total_len)
        cos = cos[past_len:total_len]
        sin = sin[past_len:total_len]

        for idx, layer in enumerate(self.layers):
            # Compute Q, K, V for this layer
            normed_x = layer.attn_norm(x)
            q = layer.attn.q_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)
            k = layer.attn.k_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)
            v = layer.attn.v_proj(normed_x).view(B, T, self.n_heads, layer.attn.head_dim).transpose(1, 2)

            q, k = layer.attn.apply_rope(q, k, cos, sin) if hasattr(layer.attn, 'apply_rope') else (q, k)

            full_k, full_v = cache.update(idx, k, v)
            scores = torch.matmul(q, full_k.transpose(-2, -1)) / (layer.attn.head_dim ** 0.5)

            if T > 1:
                total_k_len = full_k.shape[2]
                q_idx = torch.arange(total_k_len - T, total_k_len, device=x.device).unsqueeze(1)
                k_idx = torch.arange(total_k_len, device=x.device).unsqueeze(0)
                causal_mask = (k_idx > q_idx).unsqueeze(0).unsqueeze(0)
                scores = scores.masked_fill(causal_mask, float("-inf"))

            attn_weights = torch.softmax(scores, dim=-1)
            context = torch.matmul(attn_weights, full_v)
            context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
            attn_out = layer.attn.out_proj(context)

            x = x + attn_out
            x = x + layer.ffn(layer.ffn_norm(x))

        x = self.norm(x)
        return self.lm_head(x)

    def save_weights(self, path: Union[str, Path]) -> None:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)

    def load_weights(self, path: Union[str, Path]) -> None:
        self.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))


# Backward compatibility alias
AlignForgeModel = CausalTransformerModel

In [ ]:
import math
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple, Union
import torch
import torch.nn as nn



class LoRALinear(nn.Module):

    def __init__(
        self,
        base_linear: nn.Linear,
        r: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.base_linear = base_linear
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze base linear layer
        self.base_linear.weight.requires_grad = False
        if self.base_linear.bias is not None:
            self.base_linear.bias.requires_grad = False

        in_features = base_linear.in_features
        out_features = base_linear.out_features

        # Low-rank matrices
        self.lora_A = nn.Parameter(torch.empty((r, in_features)))
        self.lora_B = nn.Parameter(torch.zeros((out_features, r)))

        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_linear(x)
        lora_out = (self.dropout(x) @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + lora_out * self.scaling


class LoRAModelAdapter(nn.Module, ICausalLM):

    def __init__(
        self,
        base_model: ICausalLM,
        r: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
        target_modules: Tuple[str, ...] = ("q_proj", "v_proj"),
    ) -> None:
        super().__init__()
        assert isinstance(base_model, nn.Module), "base_model must be a PyTorch nn.Module"
        self._base_model = base_model
        self._r = r
        self._alpha = alpha
        self._dropout = dropout
        self._target_modules = target_modules

        self._attach_lora()

    @property
    def config(self) -> ModelArchitectureConfig:
        return self._base_model.config

    @property
    def base_model(self) -> ICausalLM:
        return self._base_model

    def _attach_lora(self) -> None:
        for param in self._base_model.parameters():
            param.requires_grad = False

        # In modern AlignForgeModel / CausalTransformerModel
        if hasattr(self._base_model, "layers"):
            for block in self._base_model.layers:
                attn = block.attn
                for mod_name in self._target_modules:
                    if hasattr(attn, mod_name):
                        orig_layer = getattr(attn, mod_name)
                        if isinstance(orig_layer, nn.Linear):
                            lora_layer = LoRALinear(
                                orig_layer, r=self._r, alpha=self._alpha, dropout=self._dropout
                            )
                            setattr(attn, mod_name, lora_layer)

    def forward(
        self,
        input_ids: torch.Tensor,
        kv_caches: Optional[Any] = None,
        use_cache: bool = False,
    ) -> Any:
        return self._base_model.forward(input_ids, kv_caches=kv_caches, use_cache=use_cache)

    def get_num_params(self, non_embedding: bool = False) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_parameters(self) -> int:
        return self.get_num_params()

    def get_lora_state_dict(self) -> Dict[str, torch.Tensor]:
        return {
            name: param
            for name, param in self._base_model.named_parameters()
            if "lora_" in name and param.requires_grad
        }

    def save_weights(self, path: Union[str, Path]) -> None:
        save_lora_checkpoint(self._base_model, path)

    def load_weights(self, path: Union[str, Path]) -> None:
        load_lora_checkpoint(self._base_model, path)


def apply_lora_to_model(
    model: nn.Module,
    r: int = 8,
    alpha: float = 16.0,
    dropout: float = 0.0,
    target_modules: Tuple[str, ...] = ("q_proj", "v_proj"),
) -> Dict[str, int]:
    for param in model.parameters():
        param.requires_grad = False

    replaced_count = 0
    if hasattr(model, "layers"):
        for block in model.layers:
            attn = block.attn
            for mod_name in target_modules:
                if hasattr(attn, mod_name):
                    orig_layer = getattr(attn, mod_name)
                    if isinstance(orig_layer, nn.Linear):
                        lora_layer = LoRALinear(orig_layer, r=r, alpha=alpha, dropout=dropout)
                        setattr(attn, mod_name, lora_layer)
                        replaced_count += 1

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    return {
        "trainable_params": trainable_params,
        "total_params": total_params,
        "replaced_layers": replaced_count,
    }


def get_lora_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: param
        for name, param in model.named_parameters()
        if "lora_" in name and param.requires_grad
    }


def save_lora_checkpoint(
    model: nn.Module,
    output_path: Union[str, Path],
    config: Optional[Dict] = None,
) -> None:
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "lora_state_dict": get_lora_state_dict(model),
        "config": config or {},
    }
    torch.save(payload, out)


def load_lora_checkpoint(
    model: nn.Module,
    checkpoint_path: Union[str, Path],
) -> None:
    payload = torch.load(checkpoint_path, map_location="cpu")
    lora_weights = payload["lora_state_dict"]
    model.load_state_dict(lora_weights, strict=False)

## 2. DPO Loss, Dataset, and Trainer (1:1 from src)

In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import datasets


DATASET_SOURCES = {
    "tinystories": {
        "repo_id": "roneneldan/TinyStories",
        "license": "CDLA-Sharing-1.0",
        "description": "Short narrative stories with basic English vocabulary",
        "role": "Pre-training C0",
    },
    "coedit_gec": {
        "repo_id": "grammarly/coedit",
        "license": "Apache-2.0",
        "description": "Grammar error correction source-target pairs with task instructions",
        "role": "SFT C1-full and C1-lora",
    },
    "paraphrase_dpo": {
        "repo_id": "alecccdd/paraphrasing-preferences-orpo-dpo",
        "license": "Dataset Card Review Required",
        "description": "Paraphrase preference pairs with prompt, chosen, and rejected completions",
        "role": "DPO Preference Alignment C2-dpo",
    },
    "jfleg": {
        "repo_id": "jhu-clsp/jfleg",
        "license": "CC BY-NC-SA 4.0",
        "description": "Multi-reference human fluency GEC test benchmark (4 references per sentence)",
        "role": "Held-out GEC evaluation",
    },
    "prompt_injections": {
        "repo_id": "deepset/prompt-injections",
        "license": "Evaluation subset",
        "description": "Prompt injection payloads vs benign commands",
        "role": "Supplemental input guardrail evaluation",
    },
}


def format_instruction_prompt(task: str, text: str) -> str:
    if task == "fix_grammar":
        return f"Fix the grammar in this sentence: {text.strip()}\nCorrection: "
    elif task == "paraphrase":
        return f"Paraphrase this sentence to make it more natural: {text.strip()}\nParaphrase: "
    return f"{text.strip()}\n"


class HuggingFaceDatasetRepository(IDatasetRepository):

    def __init__(self, cache_dir: Optional[Path] = None) -> None:
        self._cache_dir = cache_dir

    def get_pretrain_data(self, limit: Optional[int] = 1000) -> List[str]:
        try:
            ds = datasets.load_dataset("roneneldan/TinyStories", split=f"train[:{limit}]")
            return [row["text"] for row in ds if row.get("text")]
        except Exception:
            return [
                "Once upon a time, there was a little girl named Lily. She loved reading books.",
                "Tim had a small blue car. He drove it around the garden every afternoon.",
                "A puppy saw a bird in the tree. The puppy wagged its tail happily.",
            ]

    def get_sft_data(self, limit: Optional[int] = 1200) -> List[TextCorrectionPair]:
        raw_samples = load_coedit_gec_samples(max_samples=limit or 1200)
        return [
            TextCorrectionPair(
                source_text=item["prompt"],
                target_text=item["completion"],
            )
            for item in raw_samples
        ]

    def get_dpo_data(self, limit: Optional[int] = 852) -> List[PreferencePair]:
        raw_samples = load_paraphrase_dpo_samples(max_samples=limit or 852)
        return [
            PreferencePair(
                prompt=item["prompt"],
                chosen=item["chosen"],
                rejected=item["rejected"],
            )
            for item in raw_samples
        ]


def load_coedit_gec_samples(
    max_samples: int = 1200,
    seed: int = 42,
) -> List[Dict[str, str]]:
    try:
        raw_ds = datasets.load_dataset(
            "grammarly/coedit",
            split=f"train[:{max_samples * 2}]",
        )
        samples: List[Dict[str, str]] = []
        for row in raw_ds:
            if row.get("task") == "gec":
                src = row.get("src", "").strip()
                tgt = row.get("tgt", "").strip()
                if src and tgt:
                    samples.append({
                        "task": "fix_grammar",
                        "prompt": f"{src}\nCorrection: ",
                        "completion": tgt,
                    })
                if len(samples) >= max_samples:
                    break
        return samples
    except Exception:
        return [
            {
                "task": "fix_grammar",
                "prompt": "Fix the grammar in this sentence: She go to school every day.\nCorrection: ",
                "completion": "She goes to school every day.",
            },
            {
                "task": "fix_grammar",
                "prompt": "Fix the grammar in this sentence: They was happy about the news.\nCorrection: ",
                "completion": "They were happy about the news.",
            },
            {
                "task": "fix_grammar",
                "prompt": "Fix the grammar in this sentence: He have two dog.\nCorrection: ",
                "completion": "He has two dogs.",
            },
            {
                "task": "fix_grammar",
                "prompt": "Fix the grammar in this sentence: I does not like apple.\nCorrection: ",
                "completion": "I do not like apples.",
            },
        ]


def load_paraphrase_dpo_samples(
    max_samples: int = 852,
) -> List[Dict[str, Any]]:
    try:
        raw_ds = datasets.load_dataset(
            "alecccdd/paraphrasing-preferences-orpo-dpo",
            split="train",
        )
        pairs: List[Dict[str, Any]] = []
        for row in raw_ds:
            chosen_msg = row.get("chosen")
            rejected_msg = row.get("rejected")
            prompt_text = ""
            chosen_text = ""
            rejected_text = ""

            if isinstance(chosen_msg, list) and len(chosen_msg) >= 2:
                prompt_text = format_instruction_prompt("paraphrase", chosen_msg[0].get("content", ""))
                chosen_text = chosen_msg[1].get("content", "")
            elif isinstance(chosen_msg, str):
                chosen_text = chosen_msg
                prompt_text = format_instruction_prompt("paraphrase", row.get("prompt", row.get("input", "")))

            if isinstance(rejected_msg, list) and len(rejected_msg) >= 2:
                rejected_text = rejected_msg[1].get("content", "")
            elif isinstance(rejected_msg, str):
                rejected_text = rejected_msg

            if prompt_text and chosen_text and rejected_text:
                pairs.append({
                    "prompt": prompt_text,
                    "chosen": chosen_text,
                    "rejected": rejected_text,
                })
            if len(pairs) >= max_samples:
                break
        return pairs
    except Exception:
        return [
            {
                "prompt": "Paraphrase this sentence to make it more natural: I cannot attend the meeting today.\nParaphrase: ",
                "chosen": "I won't be able to make it to today's meeting.",
                "rejected": "Cannot go meeting today because impossible.",
            },
            {
                "prompt": "Paraphrase this sentence to make it more natural: She likes reading books in the library.\nParaphrase: ",
                "chosen": "She enjoys spending time reading at the library.",
                "rejected": "She likes books reading in library space.",
            },
        ]


def load_jfleg_evaluation_samples(
    max_samples: int = 100,
) -> List[Dict[str, Any]]:
    try:
        raw_ds = datasets.load_dataset("jhu-clsp/jfleg", split="test")
        samples: List[Dict[str, Any]] = []
        for row in raw_ds:
            sentence = row.get("sentence", "").strip()
            corrections = row.get("corrections", [])
            if sentence and corrections:
                samples.append({
                    "sentence": sentence,
                    "corrections": [c.strip() for c in corrections if c.strip()],
                })
            if len(samples) >= max_samples:
                break
        return samples
    except Exception:
        return [
            {
                "sentence": "Students must to study hard to pass test.",
                "corrections": [
                    "Students must study hard to pass the test.",
                    "Students have to study hard to pass the test.",
                    "A student must study hard to pass the test.",
                    "Students must study hard in order to pass the test.",
                ],
            }
        ]


def audit_token_length_distribution(
    samples: List[Dict[str, Any]],
    tokenizer: Any,
    context_lengths: Tuple[int, ...] = (128, 256),
) -> Dict[str, Any]:
    lengths: List[int] = []
    retained_counts = {L: 0 for L in context_lengths}

    for item in samples:
        prompt = item.get("prompt", "")
        if "chosen" in item and "rejected" in item:
            p_len = len(tokenizer.encode(prompt))
            c_len = len(tokenizer.encode(item["chosen"]))
            r_len = len(tokenizer.encode(item["rejected"]))
            total_len = p_len + max(c_len, r_len)
        elif "completion" in item:
            p_len = len(tokenizer.encode(prompt))
            comp_len = len(tokenizer.encode(item["completion"]))
            total_len = p_len + comp_len
        elif "sentence" in item:
            total_len = len(tokenizer.encode(item["sentence"]))
        else:
            continue

        lengths.append(total_len)
        for L in context_lengths:
            if total_len <= L:
                retained_counts[L] += 1

    total = len(lengths) or 1
    return {
        "total_samples_analyzed": len(lengths),
        "min_tokens": min(lengths) if lengths else 0,
        "max_tokens": max(lengths) if lengths else 0,
        "mean_tokens": sum(lengths) / total if lengths else 0,
        "retention_by_context_length": {
            str(L): {
                "retained_count": retained_counts[L],
                "retention_rate_pct": round((retained_counts[L] / total) * 100, 2),
            }
            for L in context_lengths
        },
    }


def generate_dataset_manifest(
    output_path: str | Path = "reports/dataset_manifest.json",
) -> Dict[str, Any]:
    manifest = {
        "manifest_version": "1.0.0",
        "policy": "Strict public data ingestion without synthetic labels",
        "datasets": DATASET_SOURCES,
    }
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return manifest

from abc import ABC, abstractmethod
from pathlib import Path
import time
from typing import Any, Dict, Optional
import torch
import torch.nn as nn
from torch.utils.data import DataLoader



class BaseTrainer(ITrainer, ABC):

    def __init__(
        self,
        model: nn.Module,
        dataloader: DataLoader,
        lr: float = 3e-4,
        weight_decay: float = 0.01,
        max_grad_norm: float = 1.0,
        device: Optional[torch.device] = None,
        use_amp: bool = True,
    ) -> None:
        self._device = device or (
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )
        self._model = model.to(self._device)
        self._dataloader = dataloader
        self._max_grad_norm = max_grad_norm
        self._lr = lr
        self._weight_decay = weight_decay

        trainable_params = [p for p in self._model.parameters() if p.requires_grad]
        self._optimizer = torch.optim.AdamW(
            trainable_params, lr=lr, weight_decay=weight_decay
        )

        # Mixed Precision support via CUDA AMP
        self._use_amp = use_amp and (self._device.type == "cuda")
        self._scaler = torch.cuda.amp.GradScaler(enabled=self._use_amp)

    @property
    def model(self) -> nn.Module:
        return self._model

    @property
    def device(self) -> torch.device:
        return self._device

    @property
    def optimizer(self) -> torch.optim.Optimizer:
        return self._optimizer

    @property
    def use_amp(self) -> bool:
        return self._use_amp

    def autocast_context(self):
        return torch.cuda.amp.autocast(enabled=self._use_amp, dtype=torch.float16)

    def backward_and_step(self, loss: torch.Tensor) -> None:
        if self._use_amp:
            self._scaler.scale(loss).backward()
            self._scaler.unscale_(self._optimizer)
            nn.utils.clip_grad_norm_(self._model.parameters(), self._max_grad_norm)
            self._scaler.step(self._optimizer)
            self._scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(self._model.parameters(), self._max_grad_norm)
            self._optimizer.step()

    @abstractmethod
    def train_step(self, batch: Any) -> float:
        pass

    def train(self, max_steps: int = 50, log_interval: int = 10) -> Dict[str, float]:
        self._model.train()
        step_count = 0
        final_loss = 0.0
        t0 = time.time()

        while step_count < max_steps:
            for batch in self._dataloader:
                loss_val = self.train_step(batch)
                step_count += 1
                final_loss = loss_val

                if step_count % log_interval == 0 or step_count == max_steps:
                    amp_tag = " (FP16)" if self._use_amp else ""
                    print(f"Step {step_count:4d}/{max_steps}{amp_tag} | Loss: {final_loss:.4f}")

                if step_count >= max_steps:
                    break

        elapsed = time.time() - t0
        return {
            "final_loss": final_loss,
            "steps": float(step_count),
            "elapsed_seconds": elapsed,
        }

import copy
from pathlib import Path
import time
from typing import Any, Dict, List, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import yaml



def get_batch_logps(
    logits: torch.Tensor,
    labels: torch.Tensor,
    label_pad_token_id: int = -100,
) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    loss_mask = shift_labels != label_pad_token_id
    clean_labels = shift_labels.clone()
    clean_labels[~loss_mask] = 0

    per_token_logps = torch.gather(
        shift_logits.log_softmax(-1),
        dim=2,
        index=clean_labels.unsqueeze(2),
    ).squeeze(2)

    return (per_token_logps * loss_mask).sum(-1)


def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, float]:
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = reference_chosen_logps - reference_rejected_logps
    logits = pi_logratios - ref_logratios

    losses = -F.logsigmoid(beta * logits)
    chosen_rewards = beta * (policy_chosen_logps - reference_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - reference_rejected_logps).detach()
    margin = chosen_rewards - rejected_rewards
    win_rate = (logits > 0).float().mean().item()

    return losses.mean(), margin.mean(), win_rate


def compute_dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    reference_chosen_logps: torch.Tensor,
    reference_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    loss, margin, win_rate = dpo_loss(
        policy_chosen_logps,
        policy_rejected_logps,
        reference_chosen_logps,
        reference_rejected_logps,
        beta,
    )
    chosen_rewards = beta * (policy_chosen_logps - reference_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - reference_rejected_logps).detach()
    return loss, chosen_rewards.mean(), rejected_rewards.mean()


class DPODataset(Dataset):

    def __init__(
        self,
        pairs: List[Dict[str, Any]],
        tokenizer: ByteLevelBPETokenizer,
        max_seq_len: int = 128,
    ) -> None:
        self.items: List[Dict[str, torch.Tensor]] = []
        ignore_index = -100

        for p in pairs:
            prompt = p["prompt"]
            chosen = p["chosen"]
            rejected = p["rejected"]

            prompt_ids = tokenizer.encode(prompt, add_bos=True)
            chosen_ids = tokenizer.encode(f" {chosen}", add_eos=True)
            rejected_ids = tokenizer.encode(f" {rejected}", add_eos=True)

            c_input = prompt_ids + chosen_ids
            c_label = [ignore_index] * len(prompt_ids) + list(chosen_ids)

            r_input = prompt_ids + rejected_ids
            r_label = [ignore_index] * len(prompt_ids) + list(rejected_ids)

            def pad_or_truncate(tokens: List[int], labels: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
                if len(tokens) > max_seq_len:
                    tokens = tokens[:max_seq_len]
                    labels = labels[:max_seq_len]
                elif len(tokens) < max_seq_len:
                    diff = max_seq_len - len(tokens)
                    tokens = tokens + [tokenizer.pad_token_id] * diff
                    labels = labels + [ignore_index] * diff
                return torch.tensor(tokens, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

            c_in_t, c_lbl_t = pad_or_truncate(c_input, c_label)
            r_in_t, r_lbl_t = pad_or_truncate(r_input, r_label)

            self.items.append({
                "chosen_input": c_in_t,
                "chosen_label": c_lbl_t,
                "rejected_input": r_in_t,
                "rejected_label": r_lbl_t,
            })

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return self.items[idx]


class DPOTrainer(BaseTrainer):

    def __init__(
        self,
        policy_model: nn.Module,
        ref_model: nn.Module,
        dataloader: DataLoader,
        beta: float = 0.1,
        lr: float = 5e-5,
        weight_decay: float = 0.01,
        max_grad_norm: float = 1.0,
        device: Optional[torch.device] = None,
        use_amp: bool = True,
    ) -> None:
        super().__init__(
            model=policy_model,
            dataloader=dataloader,
            lr=lr,
            weight_decay=weight_decay,
            max_grad_norm=max_grad_norm,
            device=device,
            use_amp=use_amp,
        )
        self._ref_model = ref_model.to(self.device).eval()
        for p in self._ref_model.parameters():
            p.requires_grad = False
        self._beta = beta

    @property
    def ref_model(self) -> nn.Module:
        return self._ref_model

    def train_step(self, batch: Dict[str, torch.Tensor]) -> float:
        c_in = batch["chosen_input"].to(self.device)
        c_lbl = batch["chosen_label"].to(self.device)
        r_in = batch["rejected_input"].to(self.device)
        r_lbl = batch["rejected_label"].to(self.device)

        self.optimizer.zero_grad()
        with self.autocast_context():
            # Policy forward pass
            out_c = self.model(c_in)
            out_r = self.model(r_in)
            logits_pi_c = out_c[0] if isinstance(out_c, tuple) else out_c
            logits_pi_r = out_r[0] if isinstance(out_r, tuple) else out_r

            pi_chosen_logp = get_batch_logps(logits_pi_c, c_lbl)
            pi_rejected_logp = get_batch_logps(logits_pi_r, r_lbl)

            # Frozen reference forward pass
            with torch.no_grad():
                out_ref_c = self._ref_model(c_in)
                out_ref_r = self._ref_model(r_in)
                logits_ref_c = out_ref_c[0] if isinstance(out_ref_c, tuple) else out_ref_c
                logits_ref_r = out_ref_r[0] if isinstance(out_ref_r, tuple) else out_ref_r

                ref_chosen_logp = get_batch_logps(logits_ref_c, c_lbl)
                ref_rejected_logp = get_batch_logps(logits_ref_r, r_lbl)

            loss, _, _ = compute_dpo_loss(
                pi_chosen_logp,
                pi_rejected_logp,
                ref_chosen_logp,
                ref_rejected_logp,
                beta=self._beta,
            )

        self.backward_and_step(loss)
        return float(loss.item())


def train_dpo(
    config_path: str | Path = "configs/dpo.yaml",
    sft_checkpoint: str | Path = "artifacts/checkpoints/c1_lora.pt",
    output_checkpoint: str | Path = "artifacts/checkpoints/c2_dpo.pt",
    max_steps: Optional[int] = None,
    device_override: Optional[str] = None,
    batch_size: Optional[int] = None,
    use_amp: bool = True,
) -> Dict[str, float]:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    device = torch.device(
        device_override or ("cuda" if torch.cuda.is_available() else "cpu")
    )

    tok_dir = Path("artifacts/tokenizer")
    tokenizer = ByteLevelBPETokenizer.load(tok_dir)

    def create_model() -> CausalTransformerModel:
        m = CausalTransformerModel(
            vocab_size=2048,
            d_model=256,
            n_heads=4,
            n_layers=4,
            d_ff=1024,
            max_seq_len=128,
            dropout=0.0,
        )
        apply_lora_to_model(
            m,
            r=int(cfg["lora"]["r"]),
            alpha=float(cfg["lora"]["alpha"]),
            dropout=0.0,
            target_modules=tuple(cfg["lora"]["target_modules"]),
        )
        ckpt_path = Path(sft_checkpoint)
        if ckpt_path.exists():
            load_lora_checkpoint(m, ckpt_path)
        return m

    policy_model = create_model()
    ref_model = create_model()

    bs = batch_size or (8 if device.type == "cuda" else cfg["training"]["batch_size"])
    pairs = load_paraphrase_dpo_samples(max_samples=cfg["data"]["max_samples"])
    dataset = DPODataset(pairs, tokenizer, max_seq_len=128)
    loader = DataLoader(dataset, batch_size=bs, shuffle=True)

    trainer = DPOTrainer(
        policy_model=policy_model,
        ref_model=ref_model,
        dataloader=loader,
        beta=float(cfg["dpo"]["beta"]),
        lr=float(cfg["training"]["learning_rate"]),
        weight_decay=float(cfg["training"]["weight_decay"]),
        max_grad_norm=float(cfg["training"]["max_grad_norm"]),
        device=device,
        use_amp=use_amp,
    )

    steps = max_steps or cfg["training"]["max_steps"]
    metrics = trainer.train(max_steps=steps)

    out_ckpt = Path(output_checkpoint)
    out_ckpt.parent.mkdir(parents=True, exist_ok=True)
    save_lora_checkpoint(policy_model, out_ckpt, config=cfg)

    return metrics

## 3. Train DPO Preference Policy (C2-dpo)

In [ ]:
tok_dir = Path('artifacts/tokenizer')
tokenizer = ByteLevelBPETokenizer.load(tok_dir) if (tok_dir / 'tokenizer.json').exists() else ByteLevelBPETokenizer()

def build_model():
    m = CausalTransformerModel(vocab_size=2048, d_model=256, n_heads=4, n_layers=4, d_ff=1024, max_seq_len=128, dropout=0.0)
    apply_lora_to_model(m, r=8, alpha=16.0)
    sft_ckpt = Path('artifacts/checkpoints/c1_lora.pt')
    if sft_ckpt.exists():
        load_lora_checkpoint(m, sft_ckpt)
    return m

policy_model = build_model()
ref_model = build_model()

pairs = load_paraphrase_dpo_samples(max_samples=100)
dpo_ds = DPODataset(pairs, tokenizer, max_seq_len=128)
loader = DataLoader(dpo_ds, batch_size=4, shuffle=True)

dpo_trainer = DPOTrainer(
    policy_model=policy_model,
    ref_model=ref_model,
    dataloader=loader,
    beta=0.1,
    lr=5e-5,
    device=device,
)
print('=== Training DPO Preference Alignment (C2-dpo) ===')
metrics = dpo_trainer.train(max_steps=30, log_interval=10)
save_lora_checkpoint(policy_model, 'artifacts/checkpoints/c2_dpo.pt')
print(f'DPO Alignment complete! Final loss: {metrics["final_loss"]:.4f}')